# CO5085 A1 · ResNet18 và Swin-T trên CIFAR-100

Thực nghiệm so sánh CNN với Transformer ở ba mức tinh chỉnh và khi huấn luyện từ đầu. Thiết lập bổ sung so sánh riêng hai cách khởi tạo Swin-T dưới cùng lịch học 40 epoch.

Trình tự gồm kiểm tra dữ liệu, xây dựng đối chứng, huấn luyện, chọn checkpoint trên tập xác thực và đánh giá cuối. Cấu hình mặc định dùng ba hạt giống và cần GPU CUDA hỗ trợ BF16 như RTX 3090, L4 hoặc A100.

## 1. Cấu hình và đường dẫn

Hai lịch học dùng chung bộ ảnh, hạt giống chia tập và cách tiền xử lý. Có thể chạy từng hạt giống ở một phiên GPU riêng. Mỗi phiên ghi dữ liệu kết quả vào thư mục riêng với kết quả đã công bố.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
root = next((p for p in candidates if (p / 'src/A1/protocol.py').is_file()), None)
if root is None:
    if 'google.colab' not in sys.modules:
        raise FileNotFoundError('Mở notebook từ thư mục CO5085-deep-learning.')
    root = Path('/content/CO5085-deep-learning')
    if not (root / 'src/A1/protocol.py').is_file():
        subprocess.run(['git', 'clone', 'https://github.com/dathuynh1108/CO5085-deep-learning.git',
                        str(root)], check=True)
sys.path.insert(0, str(root / 'src'))
os.environ.setdefault('OMP_NUM_THREADS', '4')
os.environ.setdefault('MKL_NUM_THREADS', '4')

settings = {'assignment1': root / 'configs/a1_experiments.json',
            'a1_swin_followup': root / 'configs/a1_swin_followup.json'}
protocol_names = os.getenv('CO5085_A1_PROTOCOLS', ','.join(settings)).split(',')
seeds = [int(seed) for seed in os.getenv('CO5085_A1_SEEDS', '42,43,44').split(',')]
output_root = Path(os.getenv('CO5085_A1_OUTPUT_ROOT', str(root / 'runs/notebooks'))).resolve()
data_root = Path(os.getenv('CO5085_A1_DATA_ROOT', str(root / 'data'))).resolve()
workers = int(os.getenv('CO5085_A1_WORKERS', '8'))
resume = os.getenv('CO5085_A1_RESUME', '0') == '1'
confirm_test = os.getenv('CO5085_A1_CONFIRM_TEST', '0') == '1'

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import Image, display
from torchvision.datasets import CIFAR100
from torchvision.datasets.utils import download_url
from A1.data import development_data, loader
from A1.engine import epoch_pass, test_run, train_run
from A1.models import build_model
from A1.protocol import load_protocol
from common.runtime import atomic_json, choose_device

device = choose_device(os.getenv('CO5085_A1_DEVICE', 'cuda'))
if device.type == 'cuda' and not torch.cuda.is_bf16_supported(including_emulation=False):
    raise RuntimeError('Thiết lập A1 cần GPU hỗ trợ BF16 phần cứng.')
if len(set(seeds)) != len(seeds) or any(seed < 0 for seed in seeds) or workers < 0:
    raise ValueError('Hạt giống phải khác nhau và số worker phải không âm.')
protocols = {name: load_protocol(settings[name]) for name in protocol_names}
output_root.mkdir(parents=True, exist_ok=True)
print(f'Thiết bị: {device} · hạt giống: {seeds} · thư mục kết quả: {output_root}')
for name, protocol in protocols.items():
    print(name, protocol['epochs'], 'epoch', len(protocol['experiments']), 'cấu hình')

## 2. Khám phá và tiền xử lý dữ liệu

CIFAR-100 gồm 100 lớp ảnh RGB kích thước 32 × 32. Tập xác thực lấy 50 ảnh mỗi lớp từ tập phát triển. Ảnh huấn luyện được cắt và lật ngẫu nhiên trước khi nội suy lên 96 × 96. Hai phép đo xác thực và kiểm tra sử dụng biến đổi cố định. Biểu đồ dưới đây kiểm tra số ảnh mỗi lớp và minh họa ảnh gốc.

In [ ]:
if not (data_root / CIFAR100.base_folder / 'meta').is_file():
    # Bản sao archive giữ nguyên checksum của Toronto.
    mirror = 'https://zenodo.org/records/10089977/files/cifar-100-python.tar.gz?download=1'
    download_url(mirror, str(data_root), filename=CIFAR100.filename, md5=CIFAR100.tgz_md5)
    CIFAR100(str(data_root), train=True, download=True)

first = protocols[protocol_names[0]]
train_data, val_data, split, data_identity = development_data(data_root, first)
pixels = train_data.dataset.data
labels = np.asarray(train_data.dataset.targets)
train_counts = np.bincount(labels[split['train']], minlength=100)
val_counts = np.bincount(labels[split['val']], minlength=100)
assert len(train_data) == 45_000 and len(val_data) == 5_000
print(f'{len(train_data):,} ảnh huấn luyện và {len(val_data):,} ảnh xác thực')
print('Trung bình RGB của tập huấn luyện:', np.round(pixels[split['train']].mean((0, 1, 2)) / 255, 4))
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(np.arange(100), train_counts, label='Huấn luyện')
ax.bar(np.arange(100), val_counts, label='Xác thực')
ax.set(xlabel='Chỉ số lớp', ylabel='Số ảnh', title='Phân bố lớp CIFAR-100')
ax.legend()
fig.tight_layout()
fig.savefig(output_root / 'a1-class-counts.png', dpi=130)
display(Image(filename=str(output_root / 'a1-class-counts.png')))
plt.close(fig)

fig, axes = plt.subplots(2, 4, figsize=(8, 4))
examples = np.random.default_rng(seeds[0]).choice(split['train'], len(axes.flat), replace=False)
for ax, index in zip(axes.flat, examples):
    ax.imshow(pixels[index])
    ax.set_title(split['classes'][labels[index]], fontsize=9)
    ax.axis('off')
fig.tight_layout()
fig.savefig(output_root / 'a1-samples.png', dpi=130)
display(Image(filename=str(output_root / 'a1-samples.png')))
plt.close(fig)

## 3. Huấn luyện và chọn checkpoint

Bộ phân loại cuối của mỗi kiến trúc có 100 đầu ra. Trong từng lượt, mô hình tính logit, tính cross-entropy, lan truyền ngược, cắt chuẩn gradient và cập nhật AdamW. Sau mỗi epoch, mô hình được đo trên tập xác thực; độ chính xác quyết định checkpoint, với cross-entropy dùng để phân giải trường hợp bằng nhau. Log theo epoch, hạt giống và cấu hình được lưu cùng checkpoint.

In [ ]:
jobs = []
for name, protocol in protocols.items():
    for experiment in protocol['experiments']:
        for seed in seeds:
            run = output_root / name / experiment['id'] / f'seed_{seed}'
            options = dict(run=run, protocol=protocol, experiment=experiment, seed=seed,
                           data_root=data_root, device=device, workers=workers, config=settings[name])
            print(f"{name} · {experiment['id']} · hạt giống {seed}", flush=True)
            metadata = train_run(**options, resume=resume)
            assert metadata['status'] == 'trained'
            jobs.append((name, options))

## 4. Đánh giá sau khi khóa cấu hình

Đánh giá kiểm tra được bật rõ ràng bằng `CO5085_A1_CONFIRM_TEST=1` sau khi cố định giao thức. Khi chưa bật, notebook tạo bảng và ma trận nhầm lẫn trên tập xác thực. Mỗi dòng kết quả gắn với hạt giống và checkpoint đã chọn.

In [ ]:
rows, matrices = [], {}
for name, options in jobs:
    run = options['run']
    preset, seed = options['experiment'], options['seed']
    record = json.loads((run / 'metadata.json').read_text())
    if confirm_test:
        metrics = test_run(**options)
    else:
        model = build_model(preset, initialize_pretrained=False).to(device)
        checkpoint = torch.load(run / 'best.pt', map_location='cpu', weights_only=False)
        model.load_state_dict(checkpoint['model'])
        batches = loader(val_data, options['protocol']['batch_size'], workers,
                         torch.Generator().manual_seed(seed + 1), False, device)
        metrics = epoch_pass(model, batches, device)
        del model, checkpoint, batches
    label = (name, preset['id'])
    matrices[label] = matrices.get(label, 0) + np.asarray(metrics['confusion'])
    rows.append({'Thiết lập': name, 'Cấu hình': preset['id'], 'Hạt giống': seed,
                 'Tập': 'Kiểm tra' if confirm_test else 'Xác thực',
                 'Epoch chọn': record['best_validation']['epoch'],
                 'Accuracy': metrics['accuracy'], 'Macro-F1': metrics['macro_f1'],
                 'Tham số học': record['trainable_params'], 'Giây huấn luyện': record['train_seconds'],
                 'VRAM MiB': record['peak_gpu_memory_bytes'] / 2**20})
results = pd.DataFrame(rows)
display(results.round(4))
display(results.groupby(['Thiết lập', 'Cấu hình'])[['Accuracy', 'Macro-F1']].agg(['mean', 'std']).round(4))
results.to_csv(output_root / f'a1-metrics-seeds-{"-".join(map(str, seeds))}.csv', index=False)
atomic_json(output_root / f'a1-config-seeds-{"-".join(map(str, seeds))}.json', {
    'seeds': seeds, 'protocols': protocols, 'data': data_identity, 'test': confirm_test,
    'runs': [str(options['run']) for _, options in jobs]})

## 5. Đường cong học và ma trận nhầm lẫn

Đường cong lấy trung bình các hạt giống đã chạy trong phiên này. Ma trận chuẩn hóa theo nhãn thực tế để quan sát tỷ lệ lỗi từng lớp. Với 100 lớp, bảng lỗi bên cạnh giúp xác định những cặp thường bị nhầm.

In [ ]:
for name, protocol in protocols.items():
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for preset in protocol['experiments']:
        histories = []
        for seed in seeds:
            history = output_root / name / preset['id'] / f'seed_{seed}' / 'history.csv'
            histories.append(pd.read_csv(history))
        for ax, metric, title in zip(axes, ['val_accuracy', 'val_ce'], ['Accuracy xác thực', 'Cross-entropy xác thực']):
            values = np.stack([history[metric] for history in histories])
            ax.plot(histories[0]['epoch'], values.mean(0), label=preset['id'])
            ax.set(xlabel='Epoch', ylabel=title)
    axes[1].legend(fontsize=7)
    fig.tight_layout()
    path = output_root / f'{name}-curves-{"-".join(map(str, seeds))}.png'
    fig.savefig(path, dpi=150)
    display(Image(filename=str(path)))
    plt.close(fig)

for name, protocol in protocols.items():
    ids = [preset['id'] for preset in protocol['experiments'] if preset['regime'] in ('scratch', 'full')]
    fig, axes = plt.subplots(1, len(ids), figsize=(4 * len(ids), 4), squeeze=False)
    mistakes = []
    for ax, preset_id in zip(axes.flat, ids):
        matrix = matrices[name, preset_id]
        proportions = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
        ax.imshow(proportions, cmap='Blues', vmin=0, vmax=1)
        ax.set(title=preset_id, xlabel='Nhãn dự đoán', ylabel='Nhãn thực tế')
        off_diagonal = matrix.copy()
        np.fill_diagonal(off_diagonal, 0)
        actual, predicted = np.unravel_index(off_diagonal.argmax(), matrix.shape)
        mistakes.append((preset_id, split['classes'][actual], split['classes'][predicted],
                         int(off_diagonal[actual, predicted])))
    fig.tight_layout()
    path = output_root / f'{name}-confusion-{"-".join(map(str, seeds))}.png'
    fig.savefig(path, dpi=150)
    display(Image(filename=str(path)))
    plt.close(fig)
    display(pd.DataFrame(mistakes, columns=['Cấu hình', 'Nhãn thực tế', 'Nhãn dự đoán', 'Số lỗi']))

## 6. Phạm vi diễn giải

Đối chứng 20 epoch phản ánh ảnh hưởng của tiền huấn luyện và mức độ tinh chỉnh trong cùng ngân sách. Đối chứng 40 epoch giữ chung tốc độ học, bộ tối ưu và số epoch giữa hai cách khởi tạo Swin-T. So sánh giữa hai lịch cần xét các thay đổi đồng thời về warmup, tốc độ học và hệ số suy giảm trọng số. Phần kết quả trong báo cáo đi kèm số liệu theo hạt giống và phân tích ảnh trùng pixel.